# 09_02 Searching both poems, and scoring the search

*📖 Not in the book*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → model learns → model writes → MCP tool
> ```
>
> You are here: **after the main course**: checking how often the librarian brings the right page
>
> Project files this connects to:
>
> - `src/homer_ai/retrieval.py`
> - `data/eval/questions.jsonl`
> - `tests/test_retrieval.py`
>
> **This notebook:** One search that worked proves nothing. How good is ours on 40 questions with known answers, and would vectors do better than counting words?
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- Why a question about Odysseus finds nothing in Butler's English, and a small fix
- How to label questions with the lines that answer them
- What recall@k and MRR are, worked out on 4 questions and then on 40
- Why our own model's embeddings can't do this search
- How a pretrained sentence model compares with BM25 (optional cell)

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **labelled question** | a question stored together with the places in the text that answer it | *Which suitor does Odysseus kill first?* → `odyssey 22:2` |
| **recall@k** | the share of questions for which a right passage is among the first k results | recall@5 = 0.55: for 22 of 40 questions it's in the first 5 |
| **MRR** | mean reciprocal rank: the average of 1/rank over all questions, with 0 when nothing right was found | ranks 1 and 2 → (1 + 0.5) / 2 = 0.75 |
| **query expansion** | adding words to a question before searching, so it matches the words the text uses | `odysseus` → `odysseus ulysses` |
| **sentence embedding** | a list of numbers for a whole sentence or paragraph, made by a model trained so that texts with a similar meaning get similar lists | 384 numbers per paragraph |

Words you've met before: **rank** ([08_04](../08_advanced_fine_tuning/08_04_lora.ipynb)).

## 3. The idea

### ❓ The search found Noman. Are we done?

No. One question that worked tells us very little. I picked it, and I picked it because it works.

It's the lesson of [05_03](../05_training_on_the_odyssey/05_03_memorising_vs_understanding.ipynb) and [05_05](../05_training_on_the_odyssey/05_05_validation.ipynb) again. There, a low loss on the training text didn't prove the model had learned, so we measured on text it hadn't seen. Here, a demo that looks good doesn't prove the search is good. We need a set of questions with known answers, and a number.

So this notebook does three things: it fixes one clear problem, builds the yardstick, and then uses the yardstick on our search and on two other ideas.

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Step 1: how do I search both poems with one call?

`search` in `retrieval.py` wraps what 09_01 built. It builds the index the first time you ask and keeps it in memory. It returns passages with their scores.

```
search(question, poem="both", language="english", k=5, names=True)
```

Let's ask about the Iliad this time. (`names=False` switches off the fix of step 2, so we see the plain search first.)

In [ ]:
from homer_ai.retrieval import search

question = "Who is the Trojan spy caught at night by Odysseus and Diomedes?"
for passage, score in search(question, k=5, names=False):
    print(f"{score:5.1f}  {passage.ref:12} {passage.text[:70]}…")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The first hit is `iliad 10:32`, where Ulysses tells the man they've just caught *Fear not*, and starts questioning him. The third, `iliad 10:29`, names him: Dolon. Right book, right scene. The last two are from other books and only share words like `night` and `Trojans`.

Both poems are in one index, so the search also decides which poem to look in.

</details>

### ❓ Step 2: why did `odysseus` have such a strange idf?

Because Butler doesn't use that name. He translated in the 1890s, when English books called the Greek gods and heroes by their Roman names. Let's count.

👉 **Predict first:** how many times does `Odysseus` appear in Butler's two translations? And `Zeus`?

In [ ]:
from homer_ai.retrieval import BUTLER_NAMES, get_index

passages, index = get_index("both", "english")  # built once, reused from here on
print(f"{'you say':12} {'passages':>8}    {'Butler says':12} {'passages':>8}")
for greek_name in ("odysseus", "zeus", "athena", "poseidon", "hera", "hermes"):
    roman = BUTLER_NAMES[greek_name]
    print(
        f"{greek_name:12} {len(index.postings.get(greek_name, [])):>8}    "
        f"{roman:12} {len(index.postings.get(roman, [])):>8}"
    )

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`Odysseus` is in no passage at all, and `Ulysses` is in 477. `Zeus` appears in one passage, `Jove` in 501. Athena is `Minerva`, Poseidon is `Neptune`, Hera is `Juno`, Hermes is `Mercury`.

So when you type *Odysseus*, the search throws the word away. No passage has it, so it adds nothing to any score.

</details>

### ❓ Step 3: how do I fix that?

With a small dictionary. Before searching, for every Greek name in the question we add Butler's name next to it. Adding words to a question so that it matches the text is called **query expansion**.

```
question words:  who  helps  odysseus          among  the  phaeacians
after expansion: who  helps  odysseus ulysses  among  the  phaeacians
```

`expand_names` does it, with the 16 names in `BUTLER_NAMES`. `search` uses it unless you say `names=False`.

👉 **Predict first:** `ulysses` is in 477 passages, so its idf is low. Will adding it change much?

In [ ]:
from homer_ai.retrieval import expand_names, first_hit_rank, load_questions, tokenize

questions = load_questions()
nausicaa = questions[12]
print(nausicaa["question"])
print(expand_names(tokenize("who helps Odysseus among the Phaeacians")))
print()
for names in (False, True):
    found = [p for p, _ in search(nausicaa["question"], k=10, names=names)]
    print(f"names={names!s:5}  first three: {[p.ref for p in found[:3]]}")
    print(f"             right passage at rank: {first_hit_rank(found, nausicaa)}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

For this question it changes a lot. Without the names, the right passage (Odysseus meeting Nausicaa in Book 6 of the Odyssey) is at rank 10. With them it's at rank 1.

The word `ulysses` is weak, but here it was enough to lift Odyssey paragraphs above others that also mention a princess or the Phaeacians.

Is that typical, or did I pick a lucky example again? One example can't tell us. Time for the yardstick.

</details>

### ❓ Step 4: what is a labelled question?

A question, plus the places in the text where the answer is. The file `data/eval/questions.jsonl` has 40 of them, 20 for each poem, one JSON object per line. I wrote the questions first, the way a reader would ask them, with the Greek names. Then I found each answer in Butler's text and wrote down its book and lines.

A place is `[book, first line, last line]` in the English file. A question can have several: Penelope's weaving trick is told three times, in Books 2, 19 and 24.

In [ ]:
print(len(questions), "questions")
for q in (questions[3], questions[1]):
    print()
    for key in ("id", "poem", "question", "answer", "refs"):
        print(f"{key:9} {q[key]}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Question 4 has one place: `[9, 24, 25]`, Book 9, lines 24 to 25. Question 2 has three.

The `answer` field is for people who read the file. The search is scored only on the places: a returned passage is **right** if it's from that poem and book and overlaps one of the line ranges. `first_hit_rank` gives the rank of the first right passage in a result, or `None`.

</details>

### ❓ Step 5: how do I turn 40 results into one number?

Two numbers are common, and both start from the rank of the first right passage for each question. Take four questions:

```
question :   A     B     C      D
rank     :   1     3   none     2
in top 1?:  yes    no    no     no     → recall@1 = 1/4 = 0.25
in top 5?:  yes   yes    no    yes     → recall@5 = 3/4 = 0.75
1 / rank :   1    0.33    0    0.5     → MRR = 1.83 / 4 = 0.46
```

**recall@k** asks: for what share of the questions is a right passage among the first `k`? It matters because of the next step of the pipeline. If we hand Claude 5 passages, recall@5 is how often the answer is on the table at all.

**MRR**, mean reciprocal rank, also rewards a right passage for being near the top: rank 1 gives 1 point, rank 2 half a point, rank 3 a third, and not found gives 0. It's one number for *how far down do I have to read?*

👉 **Predict first:** if question C were found at rank 10, which of the three numbers would change?

In [ ]:
ranks = [1, 3, None, 2]


def recall_at(k, ranks):
    return sum(rank is not None and rank <= k for rank in ranks) / len(ranks)


def mrr(ranks):
    return sum(1 / rank for rank in ranks if rank is not None) / len(ranks)


print("recall@1:", recall_at(1, ranks))
print("recall@5:", recall_at(5, ranks))
print(f"MRR:      {mrr(ranks):.3f}")
print()
ranks[2] = 10
print(f"with C at rank 10: recall@5 {recall_at(5, ranks)}, MRR {mrr(ranks):.3f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

0.25, 0.75 and 0.458, as in the picture. With C at rank 10, recall@1 and recall@5 stay where they were, and MRR creeps up from 0.458 to 0.483: a tenth of a point, shared over four questions. (recall@10 would go up too, from 0.75 to 1.0.)

`retrieval_scores` in `retrieval.py` is these two functions together.

</details>

### ❓ Step 6: so how good is our search?

We run all 40 questions twice, without and with the names, and look at the first 10 results each time.

It's the rule of [05_04](../05_training_on_the_odyssey/05_04_running_fair_experiments.ipynb): change one thing, keep everything else the same, and compare on the same questions.

👉 **Predict first:** for how many of the 40 questions will a right passage be in the first 5?

In [ ]:
from homer_ai.retrieval import retrieval_scores


def all_ranks(find):
    # find(question text) -> passages, best first
    return [first_hit_rank(find(q["question"]), q) for q in questions]


def show(name, ranks):
    s = retrieval_scores(ranks)
    print(
        f"{name:28} {s['recall@1']:>8.3f} {s['recall@5']:>8.3f} "
        f"{s['recall@10']:>9.3f} {s['mrr']:>6.3f}"
    )


print(f"{'':28} {'recall@1':>8} {'recall@5':>8} {'recall@10':>9} {'MRR':>6}")
plain_ranks = all_ranks(lambda text: [p for p, _ in search(text, k=10, names=False)])
bm25_ranks = all_ranks(lambda text: [p for p, _ in search(text, k=10)])
show("BM25, question as it is", plain_ranks)
show("BM25, plus Butler's names", bm25_ranks)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

| Search | recall@1 | recall@5 | recall@10 | MRR |
|---|---|---|---|---|
| BM25, the question as it is | 0.300 | 0.525 | 0.600 | 0.384 |
| BM25, plus Butler's names | 0.350 | 0.550 | 0.575 | 0.420 |

With the names, a right passage is first for 14 of 40 questions and in the first 5 for 22 of 40. So for 18 questions, Claude would get five passages and none of them would hold the answer.

That's the honest size of what we built: a search that works about half the time. And the names helped only a little: two more questions at rank 1, one more in the first 5, and one *fewer* in the first 10. The Nausicaa example of step 3 was the best case. This is why we count.

</details>

### ❓ Step 7: which questions does it miss, and why?

Numbers say how often. To learn why, you have to read the misses. Here are the questions with no right passage in the first 10.

In [ ]:
missed = [q for q, rank in zip(questions, bm25_ranks, strict=True) if rank is None]
print(len(missed), "questions without a right passage in the first 10")
for q in missed:
    print(f"  {q['id']:>2}  {q['question']}")
    print(f"      → {q['answer']}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

17 misses, and most have the same shape. Take *What does Circe turn Odysseus' men into?* The answer is *pigs*, and that's the one word that would find the passage. But it's the answer, so it isn't in the question. What's left is `circe`, `turn`, `men` and `ulysses`, and `circe` alone is in 34 paragraphs.

Or *Who kills Patroclus?*: the one name in it, `patroclus`, is in 124 passages. Or *How does the Iliad end?*: the only word it shares with the last paragraph is `the`.

So the limit from [09_00](09_00_chapter_overview.ipynb) is real. Our librarian matches words. When the question and the passage use different words for the same thing, the search has nothing to hold on to.

</details>

### ❓ Step 8: we have embeddings. Couldn't our own model's vectors match by meaning?

It's a fair idea. In [02_05](../02_text_to_numbers/02_05_token_embeddings.ipynb) each token got a list of numbers that *can carry meaning*. So give every passage a vector, give the question a vector, and take the passages whose vectors point the same way (the dot product of [A_03](../A_toolbox/A_03_dot_product.ipynb), scaled so that length doesn't count: **cosine similarity**, 1 for the same direction).

The trouble is what our tokens are. They're single characters. The model has a vector for `π` and one for `ο`, and none for `πολύτροπον`. The simplest way to get one vector for a text is to average the vectors of its characters.

👉 **Predict first:** `πολύτροπον` (the *man of many turns* of the Odyssey's first line) and the same ten letters scrambled, `νοπορτύλοπ`. How similar will their vectors be?

In [ ]:
import torch

from homer_ai.lessons import load_homer_gpt

tokenizer, model = load_homer_gpt()  # read-only
table = model.token_embedding.weight.detach()  # one row of numbers per character


def text_vector(text):
    # the average of the rows of its characters
    return table[torch.tensor(tokenizer.encode(text))].mean(dim=0)


word, scrambled = "πολύτροπον", "νοπορτύλοπ"
same = torch.cosine_similarity(text_vector(word), text_vector(scrambled), dim=0)
print(f"{word} and {scrambled}: similarity {same:.3f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Similarity 1.000: the two vectors are the same. An average doesn't care about order, so any text with those ten letters gets this vector. For this search, `πολύτροπον` is the same as a spelling mistake.

A vector per character knows about letters. Meaning lives in words and sentences, and nothing in our model stores a vector for those.

</details>

### ❓ Step 9: does it at least find the right passages?

Let's search the Greek Odyssey for `Πηνελόπεια`, Penelope's name, both ways: with the letter vectors, and with BM25 (which works on Greek too, since it only needs words).

In [ ]:
from homer_ai.retrieval import load_passages

greek = load_passages("odyssey", "greek")
vectors = torch.stack([text_vector(p.text) for p in greek])


def nearest(query, k=3):
    scores = torch.cosine_similarity(vectors, text_vector(query).unsqueeze(0), dim=1)
    best = torch.topk(scores, k)
    places, values = best.indices.tolist(), best.values.tolist()
    return [(greek[i], value) for i, value in zip(places, values, strict=True)]


query = "Πηνελόπεια"
results = (
    ("letter vectors", nearest(query)),
    ("BM25", search(query, poem="odyssey", language="greek", k=3)),
)
for name, found in results:
    for passage, score in found:
        has_it = "πηνελοπεια" in tokenize(passage.text)
        note = "has the word" if has_it else "no such word"
        print(f"{name:15} {passage.ref:18} {score:6.2f}  {note}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Two of the three passages nearest by letter vectors don't contain the name at all. They're passages whose mix of letters happens to resemble the mix in `Πηνελόπεια`. The third has it, which isn't surprising: a passage with the word in it shares its letters. BM25's three all have the word.

So our own embeddings are out. They were trained to help guess the next *letter*, and that's all they hold. To search by meaning you need a model trained on sentences. We won't train one, but we can borrow one, the way chapter 8 borrowed GPT-2.

</details>

### ❓ Step 10 (optional): does a pretrained sentence model beat BM25?

The model is `all-MiniLM-L6-v2`, a small and popular one: 22 million weights, about 90 MB, runs on a CPU. It turns a sentence or paragraph into a **sentence embedding** of 384 numbers, and it was trained on over a billion pairs of texts so that pairs with a similar meaning land close together. The library that loads it is `sentence-transformers`.

It's an optional extra, so the course doesn't depend on it. To install it: `uv sync --extra rag` (then restart the notebook's kernel). Without it, the next cell prints one line and stops.

The cell scores three searches on the same 40 questions: the sentence model alone, BM25, and the two together. *Together* uses **reciprocal rank fusion**: each passage gets `1 / (60 + rank)` from each search, the two are added, and the passages are sorted by the sum. A passage that both searches rank fairly high beats one that only one of them likes.

👉 **Predict first:** the sentence model matches meaning, not words. By how much will it beat BM25?

In [ ]:
import os

try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    SentenceTransformer = None

if SentenceTransformer is None:
    print("sentence-transformers isn't installed: cell skipped (uv sync --extra rag)")
elif os.environ.get("HOMER_SKIP_DOWNLOADS") == "1":
    print("HOMER_SKIP_DOWNLOADS=1: cell skipped (it downloads about 90 MB)")
else:
    encoder = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

    def vectors_of(texts):  # one row of 384 numbers per text, each of length 1
        return encoder.encode(texts, convert_to_tensor=True, normalize_embeddings=True)

    passage_vectors = vectors_of([p.text for p in passages])

    def by_meaning(text, k=50):
        best = torch.topk(passage_vectors @ vectors_of(text), k).indices.tolist()
        return [passages[i] for i in best]

    def together(text, k=10):
        points = {}
        for found in (by_meaning(text), [p for p, _ in search(text, k=50)]):
            for rank, passage in enumerate(found, start=1):
                points[passage] = points.get(passage, 0) + 1 / (60 + rank)
        return sorted(points, key=lambda p: (-points[p], passages.index(p)))[:k]

    print(f"{'':28} {'recall@1':>8} {'recall@5':>8} {'recall@10':>9} {'MRR':>6}")
    show("BM25, plus Butler's names", bm25_ranks)
    show("sentence model", all_ranks(lambda text: by_meaning(text, 10)))
    show("both together", all_ranks(together))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

If the cell was skipped, here's what it printed on my laptop (24 November 2026, `sentence-transformers` 6.3.0, about 12 seconds to turn the 2,140 passages into vectors):

| Search | recall@1 | recall@5 | recall@10 | MRR |
|---|---|---|---|---|
| BM25, the question as it is | 0.300 | 0.525 | 0.600 | 0.384 |
| BM25, plus Butler's names | 0.350 | 0.550 | 0.575 | 0.420 |
| sentence model (`all-MiniLM-L6-v2`) | 0.275 | 0.475 | 0.525 | 0.336 |
| both together | 0.275 | 0.575 | 0.650 | 0.415 |

The sentence model didn't beat BM25. It lost on every column. I didn't expect that.

Looking at the ranks, the two fail on different questions. The sentence model finds *Why is Achilles angry with Agamemnon?* at rank 1, where BM25 finds nothing. But it loses the Sirens, Scylla and Nausicaa completely, and BM25 has all three in its first two results, on the strength of words like `wax` and `phaeacians`. And it has a handicap here: it reads at most 256 tokens, so it only sees the first part of Butler's long paragraphs.

Together they get the best recall@5 and recall@10, 23 and 26 of 40, though fewer first places. That's why real systems often run both, which is called **hybrid search**.

One caution. With 40 questions, one question is 2.5 points. Differences of one or two questions are noise. *The vectors are not clearly better here* is a safe reading. *BM25 beats embeddings* would be saying too much.

</details>

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ Is 40 questions enough?

Enough to see that the search works about half the time, and to catch a change that breaks it. Not enough to rank two methods that differ by a few points. For that you'd want a few hundred, and more than one person writing them. Mine all come from one head.

### ❓ Could the labels be wrong?

Yes, in one direction: a passage I didn't list can hold the answer too. For question 1, *where is Odysseus when the poem begins*, the file lists `odyssey 1:2`, which names Calypso, and `odyssey 1:5`, which describes her island without naming her. The second one is in the file because a model answered from it, correctly, and I checked. Read the misses before you trust a score.

### ❓ Why is recall@10 lower with the names than without?

Because expansion adds a word to every question that mentions a god or a hero, and that word pulls in other paragraphs about the same person. For one question that pushed the right passage from rank 7 out of the first 10. A fix that helps most questions can still hurt one. The table shows both.

### ❓ How would I make the search better?

The misses of step 7 point the way: the question lacks the passage's words. So people add a step that rewrites the question, or has a model guess an answer first and searches for that. Others fetch 50 passages cheaply and let a stronger model re-order them, which is called reranking. Whatever you try, run the 40 questions before and after.

## 5. Recap

- Butler's English uses Roman names, so `odysseus` matches nothing. Adding `ulysses` to the question (query expansion) is a small, measurable fix.
- A labelled question stores the places that answer it. recall@k is how often a right passage is in the first k; MRR also rewards being near the top.
- On 40 questions our BM25 puts a right passage in the first 5 for 22 of them. It misses when the question and the passage don't share words.
- Our model's embeddings are per character. Averaged, they can't tell a word from its scrambled letters.
- A small pretrained sentence model did no better than BM25 here; the two together did a little better than either.

**Next:** [09_03: The search as an MCP tool, for Claude and other apps](09_03_the_search_tool.ipynb).